# 01 — Prophet

Baseline forecasting TMA per stasiun memakai Prophet, dengan stasiun lain
(dan hujan DAS stasiun target) sebagai regressor eksternal. Notebook ini berdiri sendiri: jalankan
`make preprocess` sekali dulu (datanya diambil dari BigQuery), lalu
Run All di sini.

Input-nya tabel `preprocessed_data_hourly` + `preprocessing_metadata_hourly`
hasil `make preprocess`: semua stasiun + kalender (sin/cos) untuk semua model,
ditambah akumulasi hujan DAS milik stasiun target. `USE_RAINFALL = False`
menjalankan ablation tanpa hujan; hasilnya disimpan sebagai `<model>_norain`.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

import io
import json
import pickle
import numpy as np
import pandas as pd

PROJECT_ID = "thesis-506605"
DATASET    = "tables"
LOCATION   = "US"

# False = ablation tanpa fitur hujan; hasil disimpan sebagai <model>_norain.
USE_RAINFALL = True

PREPROCESSED_TABLE = f"{PROJECT_ID}.{DATASET}.preprocessed_data_hourly"
METADATA_TABLE     = f"{PROJECT_ID}.{DATASET}.preprocessing_metadata_hourly"

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

# --- Metadata: kolom kalender/stasiun/fitur + mapping nama kolom BigQuery ---
meta_row = client.query(f"""
    SELECT payload
    FROM `{METADATA_TABLE}`
    ORDER BY saved_at DESC
    LIMIT 1
""").to_dataframe().iloc[0]
metadata = json.loads(meta_row["payload"])
station_cols  = metadata["station_cols"]
column_map    = metadata["column_map"]                 # station -> kolom BigQuery
reverse_column_map = {v: k for k, v in column_map.items()}

# Fitur: semua stasiun + kalender (sin/cos) untuk semua model, ditambah
# akumulasi hujan DAS milik stasiun target (lihat `make preprocess`).
if "base_feature_cols" not in metadata:
    raise RuntimeError("Metadata versi lama (tanpa fitur hujan/kalender sin-cos); jalankan `make preprocess` dulu.")
BASE_FEATURE_COLS    = metadata["base_feature_cols"]
RAIN_COLS_BY_STATION = metadata["rain_cols_by_station"] if USE_RAINFALL else {}
FEATURES_BY_STATION  = {s: BASE_FEATURE_COLS + RAIN_COLS_BY_STATION.get(s, []) for s in station_cols}
ALL_FEATURE_COLS     = BASE_FEATURE_COLS + [c for cols in RAIN_COLS_BY_STATION.values() for c in cols]


def features_for(target):
    """Kolom input model untuk stasiun target (urutan sama dengan serving)."""
    return FEATURES_BY_STATION[target]


# --- df_train / df_val / df_test dari tabel preprocessed_data --------------
def load_split(split):
    query = f"""
        SELECT *
        FROM `{PREPROCESSED_TABLE}`
        WHERE split = @split
        ORDER BY datetime
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("split", "STRING", split),
    ])
    df = client.query(query, job_config=job_config).to_dataframe()
    df = df.set_index("datetime").rename(columns=reverse_column_map)
    return df[ALL_FEATURE_COLS]


df_train = load_split("train")
df_val   = load_split("val")
df_test  = load_split("test")

print(f"Stations : {station_cols}")
n_rain = {s: len(c) for s, c in RAIN_COLS_BY_STATION.items()}
print(f"Features : {len(BASE_FEATURE_COLS)} dasar + fitur hujan per stasiun: {n_rain} (USE_RAINFALL={USE_RAINFALL})")
print(f"Train/Val/Test: {len(df_train):,} / {len(df_val):,} / {len(df_test):,} rows")

## Konfigurasi BigQuery — Hasil Model

Tabel hasil untuk model ini (semua di dataset `tables`):

- `prophet_val_metrics_hourly` — metrik validasi per (station, horizon, metric).
- `prophet_weights_hourly` — bobot terlatih per stasiun, disimpan sebagai `BYTES`.
- `prophet_predictions_hourly` — prediksi test-set per (station, horizon, datetime).
- `prophet_metrics_hourly` — metrik test-set per (station, horizon, metric).

Bobot ditulis ke BigQuery segera setelah training (bukan ke disk), sehingga
langkah test-set evaluation di bawah bisa memuat ulang bobot dari BigQuery
saja bila objek model tidak lagi ada di memori (mis. notebook dibuka ulang).

In [ ]:
MODEL_NAME = "prophet" if USE_RAINFALL else "prophet_norain"

VAL_METRICS_TABLE = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_val_metrics_hourly"
WEIGHTS_TABLE     = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_weights_hourly"
PREDICTIONS_TABLE = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_predictions_hourly"
METRICS_TABLE     = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_metrics_hourly"

WRITE_DISPOSITION = "WRITE_TRUNCATE"      # atau "WRITE_APPEND"


def load_df(df, table_id, schema):
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition=WRITE_DISPOSITION,
    )
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()   # tunggu selesai
    tbl = client.get_table(table_id)
    print(f"  {table_id}: {tbl.num_rows:,} rows")


print("Val metrics ->", VAL_METRICS_TABLE)
print("Weights     ->", WEIGHTS_TABLE)
print("Predictions ->", PREDICTIONS_TABLE)
print("Metrics     ->", METRICS_TABLE)

In [ ]:
# pip install prophet --quiet
from prophet import Prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

HORIZONS = [1, 3, 12]  # jam ke depan

In [ ]:
def regressors_for(target):
    """Regressor Prophet: stasiun lain + fitur hujan DAS target. Fitur kalender
    tidak dipakai karena sudah ditangani seasonality Prophet."""
    return [s for s in station_cols if s != target] + RAIN_COLS_BY_STATION.get(target, [])


def make_prophet_df(df, target_station, other_stations):
    """Konversi DataFrame ke format Prophet (ds, y, + regressors)."""
    out = df[[target_station] + other_stations].copy()
    out = out.reset_index().rename(columns={
        "datetime": "ds",
        target_station: "y"
    })
    return out


def build_and_train_prophet(df_train_p, other_stations):
    """Fit Prophet dengan semua stasiun lain sebagai regressor."""
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=True,
        seasonality_mode="additive",
    )
    for station in other_stations:
        m.add_regressor(station)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m.fit(df_train_p)
    return m


def predict_horizon(model, df_future_p, horizon):
    """
    Prediksi h jam ke depan menggunakan nilai aktual sebagai regressor.
    Karena Prophet tidak native multi-step, kita shift target h langkah
    dan predict satu titik sekaligus (direct forecasting strategy).
    """
    df_shifted = df_future_p.copy()
    df_shifted["ds"] = df_shifted["ds"] + pd.Timedelta(hours=horizon)
    forecast = model.predict(df_shifted)
    return forecast[["ds", "yhat"]].set_index("ds")["yhat"]


def compute_metrics(y_true, y_pred):
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)

    # sMAPE \u2014 symmetric, aman untuk nilai mendekati 0
    smape = np.mean(
        2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)
    ) * 100

    # NSE \u2014 Nash-Sutcliffe Efficiency
    nse = 1 - (np.sum((y_true - y_pred) ** 2) /
               np.sum((y_true - np.mean(y_true)) ** 2 + 1e-8))

    return {"RMSE": rmse, "MAE": mae, "sMAPE": smape, "NSE": nse}

In [ ]:
results = {}
models  = {}

for target in station_cols:
    other_stations = regressors_for(target)
    print(f"\n{'='*50}")
    print(f"Target: {target}")
    print(f"{'='*50}")

    df_train_p = make_prophet_df(df_train, target, other_stations)
    df_val_p   = make_prophet_df(df_val,   target, other_stations)

    model = build_and_train_prophet(df_train_p, other_stations)
    models[target] = model

    results[target] = {}

    for h in HORIZONS:
        preds = predict_horizon(model, df_val_p, horizon=h)

        y_true_index = df_val.index + pd.Timedelta(hours=h)
        y_true = df_val[target].reindex(y_true_index).values
        y_pred = preds.reindex(y_true_index).values

        metrics = compute_metrics(y_true, y_pred)
        results[target][f"h+{h}"] = metrics

        print(f"  h+{h:>2}h \u2192 RMSE={metrics['RMSE']:.4f} | "
              f"MAE={metrics['MAE']:.4f} | "
              f"sMAPE={metrics['sMAPE']:.2f}% | "
              f"NSE={metrics['NSE']:.4f}")

In [ ]:
weights_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("weights", "BYTES", mode="REQUIRED"),
    bigquery.SchemaField("n_bytes", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("saved_at", "TIMESTAMP", mode="REQUIRED"),
]
val_metric_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("metric", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
]

saved_at = pd.Timestamp.now(tz="UTC").tz_localize(None)

weight_rows = []
val_metric_rows = []
for station, model in models.items():
    blob = pickle.dumps(model)
    weight_rows.append({
        "model": MODEL_NAME, "station": station, "horizon": None,
        "weights": blob, "n_bytes": len(blob), "saved_at": saved_at,
    })
    for hkey, metrics in results[station].items():
        h = int(hkey.replace("h+", ""))
        for metric_name, value in metrics.items():
            val_metric_rows.append({
                "model": MODEL_NAME, "station": station, "horizon": h,
                "metric": metric_name, "value": float(value),
            })

weights_df = pd.DataFrame(weight_rows)
val_metrics_df = pd.DataFrame(val_metric_rows)
total_mb = weights_df["n_bytes"].sum() / 1e6
print(f"Menyiapkan {len(weights_df)} baris bobot ({total_mb:.2f} MB total).")

load_df(weights_df, WEIGHTS_TABLE, weights_schema)
load_df(val_metrics_df, VAL_METRICS_TABLE, val_metric_schema)
print(f"\n\u2714 Bobot + metrik validasi '{MODEL_NAME}' tersimpan di BigQuery.")

## Test Set Evaluation

Prophet dievaluasi pada test set (periode paling akhir, belum pernah
disentuh) memakai strategi direct forecasting yang sama seperti val set.

In [ ]:
def _load_weights_from_bq(station):
    query = f"""
        SELECT weights
        FROM `{WEIGHTS_TABLE}`
        WHERE station = @station
        ORDER BY saved_at DESC
        LIMIT 1
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("station", "STRING", station),
    ])
    row = client.query(query, job_config=job_config).to_dataframe()
    if row.empty:
        raise RuntimeError(f"Bobot Prophet utk '{station}' tidak ditemukan di {WEIGHTS_TABLE}.")
    return row.iloc[0]["weights"]


def predict_test_prophet(target, h):
    if target in models:
        model = models[target]
    else:
        # rebuild dari BigQuery (mis. notebook baru dibuka ulang)
        model = pickle.loads(_load_weights_from_bq(target))

    other = regressors_for(target)
    df_test_p = make_prophet_df(df_test, target, other)
    preds = predict_horizon(model, df_test_p, horizon=h)

    y_true_index = df_test.index + pd.Timedelta(hours=h)
    y_true = df_test[target].reindex(y_true_index).values
    y_pred = preds.reindex(y_true_index).values
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    return y_true_index[mask], y_true[mask], y_pred[mask]


test_results = {}
test_predictions = []   # baris panjang utk BigQuery

for target in station_cols:
    test_results[target] = {}
    for h in HORIZONS:
        idx, y_true, y_pred = predict_test_prophet(target, h)
        metrics = compute_metrics(y_true, y_pred)
        test_results[target][f"h+{h}"] = metrics

        for ts, yt, yp in zip(idx, y_true, y_pred):
            test_predictions.append({
                "model": MODEL_NAME,
                "station": target,
                "horizon": int(h),
                "target_datetime": pd.Timestamp(ts),
                "y_true": float(yt),
                "y_pred": float(yp),
            })

    row = " | ".join(
        f"h+{h}: NSE={test_results[target][f'h+{h}']['NSE']:.3f}"
        for h in HORIZONS
    )
    print(f"{target:<20} {row}")

test_pred_df = pd.DataFrame(test_predictions)
print(f"\nTest predictions: {test_pred_df.shape[0]:,} rows.")
test_pred_df.head()

In [ ]:
metric_rows = []
for station, horizons in test_results.items():
    for hkey, metrics in horizons.items():
        for metric_name, value in metrics.items():
            metric_rows.append({
                "model": MODEL_NAME, "station": station,
                "horizon": int(hkey.replace("h+", "")),
                "metric": metric_name, "value": float(value),
            })
test_metrics_df = pd.DataFrame(metric_rows)
print(f"Test metrics rows: {len(test_metrics_df)}")

## Write Results to BigQuery

Prediksi dan metrik test-set model ini ditulis ke BigQuery. Autentikasi
mengikuti mekanisme standar Google Cloud (Application Default Credentials,
mis. kredensial bawaan Colab). Mode tulis default `WRITE_TRUNCATE` (ganti
seluruh isi tabel); ubah `WRITE_DISPOSITION` di atas ke `WRITE_APPEND` bila
ingin menumpuk hasil beberapa run.

In [ ]:
pred_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("target_datetime", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("y_true", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("y_pred", "FLOAT", mode="NULLABLE"),
]
metric_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("metric", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
]

# target_datetime -> pastikan tz-naive UTC agar konsisten di BQ
_pred = test_pred_df.copy()
_pred["target_datetime"] = pd.to_datetime(_pred["target_datetime"])

print("Uploading to BigQuery ...")
load_df(_pred, PREDICTIONS_TABLE, pred_schema)
load_df(test_metrics_df, METRICS_TABLE, metric_schema)
print("\n\u2714 BigQuery write selesai.")

## Interpretasi Model (XAI)

Prophet adalah model **aditif**, sehingga bisa dijelaskan secara eksak tanpa
aproksimasi: untuk setiap titik test,

`yhat = trend + seasonality (harian/mingguan/tahunan) + Σ efek regressor`

Dari dekomposisi ini (dihitung pada **test set**, skala terpreprocess 0–1) diturunkan:

1. **Porsi variansi prediksi per komponen** — `Cov(komponen, yhat) / Var(yhat)`.
   Ini dekomposisi variansi yang eksak: semua komponen berjumlah 100%. Nilai negatif
   berarti komponen itu justru meredam sebagian gerakan komponen lain. Kolom `std_contrib`
   dan `mean_contrib` melengkapi dengan besar gerakan dan pergeseran rata-ratanya.
2. **Efek regressor** — koefisien Prophet dikali simpangan baku regressor di train set:
   perubahan prediksi bila regressor naik 1 simpangan baku (tanda = arah pengaruh,
   mis. TMA hulu naik → TMA target naik).
3. **Penjelasan lokal** — waterfall untuk titik dengan TMA tertinggi di test set:
   komponen mana yang membuat prediksi di atas/bawah rata-ratanya.

**Cara membaca & batasan**
- Pada strategi *direct forecasting* di notebook ini, regressor diambil pada waktu `t`
  sedangkan `ds` digeser ke `t+h`. Akibatnya efek regressor **sama untuk semua horizon**;
  yang berubah antar-horizon hanya trend dan seasonality. Ini sifat model, bukan galat XAI.
- Efek regressor bersifat linear dan global (satu koefisien per stasiun), berbeda dengan
  LSTM/GRU yang dapat berubah menurut kondisi. Regressor yang berkorelasi kuat
  (mis. TMA beberapa stasiun, atau akumulasi hujan 3 jam vs 6 jam) membagi pengaruh,
  sehingga koefisien individual perlu dibaca dengan hati-hati.
- Importance menunjukkan apa yang *dipakai model*, bukan hubungan sebab-akibat di DAS.
- Bagian ini ditaruh setelah penulisan ke BigQuery sehingga kegagalan XAI tidak
  memengaruhi hasil yang sudah tersimpan.


In [ ]:
import matplotlib.pyplot as plt
from prophet.utilities import regressor_coefficients

# --- Konfigurasi XAI -----------------------------------------------------------
XAI_STATIONS      = station_cols     # isi subset stasiun untuk mempercepat
XAI_TOP_K         = 10
XAI_LOCAL_HORIZON = HORIZONS[1]      # horizon utk penjelasan lokal (event puncak)

SEASONAL_COMPONENTS = ("daily", "weekly", "yearly")
GROUP_COLORS = {"Trend": "#7b3294", "Musiman": "#ff7f0e",
                "TMA stasiun lain": "#1f77b4", "Hujan DAS": "#2ca02c"}


def component_group(name):
    if name == "trend":
        return "Trend"
    if name in SEASONAL_COMPONENTS:
        return "Musiman"
    if name.startswith("rain_"):
        return "Hujan DAS"
    return "TMA stasiun lain"


def get_prophet_model(target):
    """Model terlatih dari memori, atau dari BigQuery bila notebook dibuka ulang."""
    return models[target] if target in models else pickle.loads(_load_weights_from_bq(target))


def decompose_forecast(model, df_future_p, horizon):
    """Prediksi test (strategi sama dengan predict_horizon) beserta kontribusi tiap
    komponen. Mode aditif -> yhat = trend + seasonality + sum(efek regressor)."""
    df_shifted = df_future_p.copy()
    df_shifted["ds"] = df_shifted["ds"] + pd.Timedelta(hours=horizon)
    n_unc, model.uncertainty_samples = model.uncertainty_samples, 0   # tanpa interval -> jauh lebih cepat
    try:
        fc = model.predict(df_shifted)
    finally:
        model.uncertainty_samples = n_unc
    cols = (["trend"] + [c for c in SEASONAL_COMPONENTS if c in fc.columns]
            + list(model.extra_regressors))
    comp = fc[cols].copy()
    comp.index = pd.DatetimeIndex(fc["ds"])
    comp["yhat"] = fc["yhat"].values
    return comp


# --- Dekomposisi prediksi test set per (stasiun, horizon) ------------------------
comp_store, xai_rows = {}, []
for target in XAI_STATIONS:
    model = get_prophet_model(target)
    df_test_p = make_prophet_df(df_test, target, regressors_for(target))
    for h in HORIZONS:
        comp = decompose_forecast(model, df_test_p, h).dropna()
        comp_cols = [c for c in comp.columns if c != "yhat"]

        err = (comp[comp_cols].sum(axis=1) - comp["yhat"]).abs().max()
        if err > 1e-6:
            print(f"! {target} h+{h}: komponen tidak menjumlah ke yhat (maks selisih {err:.2e})")

        var_yhat = comp["yhat"].var()
        for c in comp_cols:
            xai_rows.append({
                "model": MODEL_NAME, "station": target, "horizon": h,
                "component": c, "group": component_group(c),
                "mean_contrib": comp[c].mean(),     # pergeseran rata-rata (skala TMA 0-1)
                "std_contrib": comp[c].std(),       # seberapa besar komponen menggerakkan prediksi
                # Cov(c, yhat)/Var(yhat): dekomposisi variansi eksak, jumlah semua komponen = 1
                "variance_share": comp[c].cov(comp["yhat"]) / var_yhat,
            })
        comp_store[(target, h)] = comp
    print(f"{target:<20} selesai")

xai_df = pd.DataFrame(xai_rows)
print(f"\nxai_df: {len(xai_df)} baris (model, station, horizon, component)")

# --- Koefisien regressor (efek per +1 simpangan baku regressor, skala TMA 0-1) ---
coef_rows = []
for target in XAI_STATIONS:
    coefs = regressor_coefficients(get_prophet_model(target))
    for _, r in coefs.iterrows():
        coef_rows.append({
            "model": MODEL_NAME, "station": target, "regressor": r["regressor"],
            "group": component_group(r["regressor"]),
            "coef": float(r["coef"]),
            "effect_per_std": float(r["coef"] * df_train[r["regressor"]].std()),
        })
coef_df = pd.DataFrame(coef_rows)
coef_df.sort_values("effect_per_std", key=abs, ascending=False).head(10)


In [ ]:
# --- Plot global ---------------------------------------------------------------
legend_handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in GROUP_COLORS.values()]

# 1) Porsi variansi prediksi per kelompok komponen (jumlah semua kelompok = 100%)
group_share = (xai_df.groupby(["station", "horizon", "group"])["variance_share"].sum()
                     .unstack("group").reindex(columns=list(GROUP_COLORS)).fillna(0))
n_cols = 3
n_rows = int(np.ceil(len(XAI_STATIONS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.2 * n_cols, 3.4 * n_rows), squeeze=False, sharey=True)
for ax in axes.ravel():
    ax.axis("off")
w = 0.8 / len(HORIZONS)
for ax, target in zip(axes.ravel(), XAI_STATIONS):
    ax.axis("on")
    gs = group_share.loc[target]
    for j, h in enumerate(HORIZONS):
        ax.bar(np.arange(len(GROUP_COLORS)) + j * w, gs.loc[h].values, w, label=f"h+{h}")
    ax.set_xticks(np.arange(len(GROUP_COLORS)) + w * (len(HORIZONS) - 1) / 2)
    ax.set_xticklabels(list(GROUP_COLORS), rotation=20, ha="right", fontsize=8)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_title(target, fontsize=10)
axes.ravel()[0].legend(frameon=False)
fig.suptitle(f"{MODEL_NAME.upper()} — porsi variansi prediksi per kelompok komponen")
fig.tight_layout()
plt.show()

# 2) Per stasiun: komponen individual terpenting (porsi variansi), per horizon
def barh_top_k(ax, d, col):
    d = d.reindex(d[col].abs().nlargest(XAI_TOP_K).index).iloc[::-1]
    ax.barh(d["component"], d[col], color=[GROUP_COLORS[g] for g in d["group"]])
    ax.axvline(0, color="k", lw=0.6)
    ax.tick_params(axis="y", labelsize=8)


for target in XAI_STATIONS:
    fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5.2 * len(HORIZONS), 4), squeeze=False)
    for ax, h in zip(axes[0], HORIZONS):
        d = xai_df[(xai_df.station == target) & (xai_df.horizon == h)]
        barh_top_k(ax, d, "variance_share")
        ax.set_title(f"h+{h}  |  porsi variansi prediksi", fontsize=10)
    fig.legend(legend_handles, list(GROUP_COLORS), loc="lower center", ncol=len(GROUP_COLORS), frameon=False)
    fig.suptitle(f"{MODEL_NAME.upper()} — komponen terpenting: {target}")
    fig.tight_layout(rect=[0, 0.08, 1, 0.94])
    plt.show()

# 3) Arah & besar pengaruh regressor: efek prediksi per kenaikan 1 simpangan baku regressor
n_cols = 3
n_rows = int(np.ceil(len(XAI_STATIONS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.2 * n_cols, 3.6 * n_rows), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for ax, target in zip(axes.ravel(), XAI_STATIONS):
    ax.axis("on")
    d = coef_df[coef_df.station == target]
    d = d.reindex(d["effect_per_std"].abs().nlargest(XAI_TOP_K).index).iloc[::-1]
    ax.barh(d["regressor"], d["effect_per_std"], color=[GROUP_COLORS[g] for g in d["group"]])
    ax.axvline(0, color="k", lw=0.6)
    ax.tick_params(axis="y", labelsize=8)
    ax.set_title(target, fontsize=10)
fig.legend(legend_handles[2:], list(GROUP_COLORS)[2:], loc="lower center", ncol=2, frameon=False)
fig.suptitle(f"{MODEL_NAME.upper()} — efek regressor per +1 simpangan baku (skala TMA 0–1)")
fig.tight_layout(rect=[0, 0.04, 1, 0.96])
plt.show()

# 4) Ringkasan: 3 komponen teratas per (stasiun, horizon)
summary = (xai_df.assign(abs_share=xai_df["variance_share"].abs())
                 .sort_values("abs_share", ascending=False)
                 .groupby(["station", "horizon"])["component"]
                 .agg(lambda s: ", ".join(s.head(3)))
                 .rename("top3_komponen"))
summary


In [ ]:
# --- Penjelasan lokal: waterfall untuk TMA tertinggi di test set -----------------
# yhat(t) = rata-rata yhat test + jumlah (kontribusi_komponen(t) - rata-rata komponen).
# Batang kanan = komponen yang mendorong prediksi di atas rata-rata, kiri = di bawah.
def waterfall(ax, comp, t, top=8):
    dev = (comp.loc[t] - comp.mean()).drop("yhat")
    base = comp["yhat"].mean()
    keep = dev.abs().sort_values(ascending=False).index[:top]
    items = [(c, dev[c]) for c in keep]
    if len(dev) > top:
        items.append(("lainnya", dev.drop(keep).sum()))
    left = base
    for i, (name, v) in enumerate(items):
        color = "#bbbbbb" if name == "lainnya" else GROUP_COLORS[component_group(name)]
        ax.barh(i, v, left=left, color=color)
        left += v
    ax.axvline(base, ls=":", color="k", lw=0.8)
    ax.axvline(comp.loc[t, "yhat"], color="k", lw=1)
    ax.set_yticks(range(len(items)))
    ax.set_yticklabels([n for n, _ in items], fontsize=8)
    ax.invert_yaxis()


n_cols = 2
n_rows = int(np.ceil(len(XAI_STATIONS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7.5 * n_cols, 3.4 * n_rows), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")

for ax, target in zip(axes.ravel(), XAI_STATIONS):
    comp = comp_store[(target, XAI_LOCAL_HORIZON)]
    y_true = df_test[target].reindex(comp.index)
    if y_true.notna().sum() == 0:
        continue
    t_peak = y_true.idxmax()
    ax.axis("on")
    waterfall(ax, comp, t_peak)
    ax.set_title(f"{target}  |  {t_peak:%Y-%m-%d %H:%M} (h+{XAI_LOCAL_HORIZON})\n"
                 f"aktual={y_true[t_peak]:.3f}  prediksi={comp.loc[t_peak, 'yhat']:.3f}  "
                 f"(garis putus = rata-rata prediksi)", fontsize=9)

fig.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in GROUP_COLORS.values()], list(GROUP_COLORS),
           loc="lower center", ncol=len(GROUP_COLORS), frameon=False)
fig.suptitle(f"{MODEL_NAME.upper()} — penjelasan lokal (dekomposisi aditif) untuk TMA puncak di test set")
fig.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.show()
